# 03 — Backtracking

## Why This Matters for DSA
Backtracking is what you reach for whenever a problem asks for "all possible X" — all permutations, all combinations, all valid placements, all paths through a grid — under some set of constraints. It's not a new mechanism on top of recursion; it's the include/exclude pattern from `02_Recursion`, generalized from 2 choices per step to N choices per step, with one habit added: undo each choice before trying the next one. This notebook closes out Phase 01 by turning that habit into a template you can apply to a new problem in minutes instead of re-deriving it each time.

## Prerequisites
- `02_Recursion` — specifically Section 5 (Multiple Recursive Calls — The Include/Exclude Pattern), which this notebook builds on directly
- `01_Complexity_Analysis` — for recognizing O(n!) and O(2ⁿ) shapes in backtracking call trees

## Learning Objectives
By the end of this notebook, you will be able to:
- Apply the choose/explore/unchoose template to a new problem from scratch
- Generate all permutations and all combinations of a set, and explain why their algorithms differ (a `used[]` tracker vs a `start` index)
- Add pruning to a backtracking solution and explain what pruning does and doesn't change about the result
- Solve constraint-satisfaction problems like N-Queens by checking constraints incrementally, one placement at a time
- Apply backtracking to a 2D grid (word search), including the grid-specific bookkeeping of marking and unmarking visited cells
- Judge whether backtracking is actually the right approach for a given problem, versus a case where it will time out no matter how cleverly it's pruned

## Checklist Before Moving On
- [ ] I can write the choose/explore/unchoose template from memory
- [ ] I know when to use a `used[]` array (permutations) versus a `start` index (combinations) to avoid repeats
- [ ] I can add a pruning check to cut off a doomed branch before recursing into it
- [ ] I can explain why N-Queens' diagonal check works with a simple absolute-difference comparison
- [ ] I understand backtracking on a grid needs to unmark a cell as unvisited before returning, just like popping from a path vector
- [ ] I can estimate whether a backtracking approach will realistically finish in time for a given input size

## Table of Contents
1. The Choose/Explore/Unchoose Template
2. Permutations — Generating All Orderings
3. Combinations — Choosing k of n
4. Pruning — Cutting Off Doomed Branches Early
5. N-Queens — Constraint Satisfaction
6. Word Search — Backtracking on a Grid
7. When Backtracking Is (and Isn't) the Right Tool
8. Phase Checkpoint, Cheat Sheet, and Answer Key


## 1. The Choose/Explore/Unchoose Template

### The Why
Nearly every backtracking problem you'll ever meet fits the same three-step shape. Once this template is automatic, "new" backtracking problems stop feeling new — they're just this template with a different notion of "choice" and a different check for what counts as a complete or valid solution.

### Core Mechanism
- **Choose:** make a decision — add something to the current partial solution.
- **Explore:** recurse, exploring everything that can follow from this choice.
- **Unchoose:** undo the decision (this is the "backtrack") before trying the next option — so the current-solution state is clean and correct for the next sibling branch.
- This is a direct generalization of the include/exclude pattern from `02_Recursion` — instead of exactly 2 choices (include/exclude) at every step, a `for` loop over all available options lets you branch into any number of choices per step.
- The traced example below makes the choose → explore → unchoose rhythm visible: notice every "choose" has a matching "unchoose" at the same indentation level, and the recursive call sits in between them.

### Common Pitfalls
- Forgetting the unchoose step — this is the single most common backtracking bug. Without it, the shared `currentPath` (or `used[]`, or `visited[][]`) leaks state from one branch into the next, corrupting every result generated after the first mistake.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// The universal backtracking template has exactly three moves, repeated at every step:
//   1. CHOOSE   -- make a decision (add something to the current partial solution)
//   2. EXPLORE  -- recurse, exploring everything that follows FROM this choice
//   3. UNCHOOSE -- undo the decision (backtrack) before trying the NEXT option
//
// This is exactly the include/exclude pattern from 02_Recursion, generalized: instead of
// exactly 2 choices (include/exclude) per step, you can have ANY number of choices per step.

void backtrackTemplate(vector<int>& choicesAvailable, vector<int>& currentPath, int depth) {
    string indent(depth * 2, ' ');

    if (depth == (int)choicesAvailable.size()) {
        cout << indent << "COMPLETE: { ";
        for (int x : currentPath) cout << x << " ";
        cout << "}\n";
        return;
    }

    for (int option = 0; option < 2; option++) {   // "2 choices" is just this example -- could be N
        cout << indent << "choose option " << option << " at depth " << depth << "\n";
        currentPath.push_back(option);        // 1. CHOOSE

        backtrackTemplate(choicesAvailable, currentPath, depth + 1);   // 2. EXPLORE

        currentPath.pop_back();               // 3. UNCHOOSE -- undo before trying the next option
        cout << indent << "unchoose option " << option << " at depth " << depth << "\n";
    }
}

int main() {
    vector<int> choicesAvailable(2);   // depth 2, just to keep the trace short and readable
    vector<int> currentPath;
    backtrackTemplate(choicesAvailable, currentPath, 0);

    // watch the pattern: choose -> explore (recurse) -> unchoose, repeated. Every "COMPLETE"
    // line is one full path through the decision tree -- one leaf. The choose/unchoose pairs
    // around each recursive call are what let the SAME currentPath vector be reused across
    // every branch, instead of allocating a fresh copy per branch.

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 2. Permutations — Generating All Orderings

### The Why
Permutations are the most direct application of the template: at every position, try every value that hasn't been placed yet.

### Core Mechanism
- A `used[]` boolean array tracks which values are already part of the current path — at each position, skip any value already used, try every value that isn't.
- Choose: mark the value used, append it to the current path. Explore: recurse to fill the next position. Unchoose: remove it from the path, mark it unused again — making it available for other branches.
- **Complexity:** the first position has `n` choices, the second has `n-1` (one value is already used), the third has `n-2`, and so on — total leaves in the decision tree is `n × (n-1) × (n-2) × ... × 1 = n!`. This is the O(n!) complexity class from `01_Complexity_Analysis`, now visible as the direct shape of the decision tree rather than an abstract formula.

### Common Pitfalls
- Using a `start` index (like combinations, next section) instead of a `used[]` array — this would generate each set of values only once per relative ORDER, silently turning your permutation generator into a combination generator.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// generate all permutations of nums using a "used[]" tracker: at each position, try every
// value that hasn't been used yet in the current path
void permute(vector<int>& nums, vector<bool>& used, vector<int>& current, vector<vector<int>>& result) {
    if ((int)current.size() == (int)nums.size()) {
        result.push_back(current);   // full-length path -- one complete permutation
        return;
    }

    for (int i = 0; i < (int)nums.size(); i++) {
        if (used[i]) continue;              // skip values already placed in this path

        used[i] = true;                     // CHOOSE
        current.push_back(nums[i]);

        permute(nums, used, current, result);   // EXPLORE

        current.pop_back();                 // UNCHOOSE
        used[i] = false;
    }
}

int main() {
    vector<int> nums{1, 2, 3};
    vector<bool> used(nums.size(), false);
    vector<int> current;
    vector<vector<int>> result;

    permute(nums, used, current, result);

    cout << "permutations of {1,2,3}:\n";
    for (auto &perm : result) {
        cout << "{ ";
        for (int x : perm) cout << x << " ";
        cout << "}\n";
    }
    cout << "total=" << result.size() << " (3! = 6)\n";

    // complexity: n choices at the first position, n-1 at the second, n-2 at the third, ...
    // total leaves = n! -- this IS the O(n!) complexity class from 01_Complexity_Analysis,
    // now seen as the direct result of a backtracking decision tree's shape

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. Combinations — Choosing k of n

### The Why
Combinations look similar to permutations at first glance but need a different bookkeeping trick — order doesn't matter, so `{1,2}` and `{2,1}` should count as the SAME combination, not two different ones. Using `used[]` here would generate duplicates; the fix is a `start` index instead.

### Core Mechanism
- Instead of tracking "which values have been used" (any order), track "the smallest index I'm still allowed to consider" — and always recurse starting from ONE PAST the index just chosen (`i + 1`), never going backward.
- This guarantees every combination is generated in increasing order exactly once — there's no way to reach `{2,1}` because once you've moved past index 1 (value 2), you can never go back to index 0 (value 1).
- Stop condition: once the current combination reaches size `k`, record it and return — no need to keep exploring further from a complete combination.

### Common Pitfalls
- Starting the inner loop from `0` instead of `start` — this reintroduces the "different orders of the same set" duplication that the `start` index exists specifically to prevent.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// choose k elements out of n, order doesn't matter -- the key difference from permutations:
// once we move past an index, we never come back to it (no "used[]" needed, just a start index)
void combine(int n, int k, int start, vector<int>& current, vector<vector<int>>& result) {
    if ((int)current.size() == k) {
        result.push_back(current);
        return;
    }

    // start from `start`, not 0 -- this is what guarantees each combination is generated
    // exactly once, in increasing order, instead of once per possible ORDERING (which would
    // be permutations, not combinations)
    for (int i = start; i <= n; i++) {
        current.push_back(i);                          // CHOOSE
        combine(n, k, i + 1, current, result);          // EXPLORE -- next call starts AFTER i
        current.pop_back();                             // UNCHOOSE
    }
}

int main() {
    vector<int> current;
    vector<vector<int>> result;
    combine(4, 2, 1, current, result);   // choose 2 elements from {1,2,3,4}

    cout << "combinations of 2 from {1,2,3,4}:\n";
    for (auto &c : result) {
        cout << "{ ";
        for (int x : c) cout << x << " ";
        cout << "}\n";
    }
    cout << "total=" << result.size() << " (C(4,2) = 6)\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. Pruning — Cutting Off Doomed Branches Early

### The Why
Backtracking's brute-force honesty (try everything) is also its biggest weakness — most non-trivial problems have far too many total branches to explore blindly. Pruning is how you make backtracking practical: recognize a branch CAN'T possibly lead to a valid solution, and skip exploring it entirely, rather than wasting time recursing into it only to fail deep inside.

### Core Mechanism
- **Unpruned:** explore every combination fully, only checking whether it's a valid solution once you've built the whole thing (or gone too far). This wastes time on branches that were already doomed from an early step.
- **Pruned:** check the constraint AS EARLY AS POSSIBLE — often right before making a choice, not after. In `combinationSumPruned`, sorting the candidates first means once one candidate would push the sum over the target, EVERY later candidate (being equal or bigger) would too — so the loop can `break` immediately instead of even checking them.
- **The core guarantee of pruning: it changes HOW MUCH WORK is done, never WHAT the answer is.** A correctly pruned backtracking solution finds exactly the same set of results as the unpruned version — just faster, because it stops wasting time exploring paths that were mathematically guaranteed to fail.

### Common Pitfalls
- Pruning with a condition that's too aggressive — accidentally cutting off a branch that COULD have led to a valid solution, silently producing wrong (incomplete) results instead of just slower-but-correct ones. Always double check a pruning condition is a genuine mathematical guarantee of failure, not just "usually" a bad sign.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <algorithm>
using namespace std;

long long callsUnpruned = 0;
long long callsPruned = 0;

// WITHOUT pruning: explore every combination, check the sum only at the very end.
// Wastes huge amounts of time exploring branches that were already doomed.
void combinationSumUnpruned(vector<int>& candidates, int target, int start,
                             vector<int>& current, int currentSum, vector<vector<int>>& result) {
    callsUnpruned++;
    if (currentSum == target) {
        result.push_back(current);
        return;
    }
    if (currentSum > target || start == (int)candidates.size()) return;   // only checked AFTER overshooting

    for (int i = start; i < (int)candidates.size(); i++) {
        current.push_back(candidates[i]);
        combinationSumUnpruned(candidates, target, i, current, currentSum + candidates[i], result);
        current.pop_back();
    }
}

// WITH pruning: sort the candidates first, then STOP exploring a branch the moment adding
// the next candidate would exceed the target -- since candidates are sorted, every candidate
// after this one is even bigger, so there's no point checking them at all.
void combinationSumPruned(vector<int>& candidates, int target, int start,
                           vector<int>& current, int currentSum, vector<vector<int>>& result) {
    callsPruned++;
    if (currentSum == target) {
        result.push_back(current);
        return;
    }

    for (int i = start; i < (int)candidates.size(); i++) {
        if (currentSum + candidates[i] > target) break;   // PRUNE: sorted, so break not continue --
                                                             // every later candidate is >= this one
        current.push_back(candidates[i]);
        combinationSumPruned(candidates, target, i, current, currentSum + candidates[i], result);
        current.pop_back();
    }
}

int main() {
    vector<int> candidates{2, 3, 5, 7};
    int target = 7;

    vector<int> current1;
    vector<vector<int>> result1;
    combinationSumUnpruned(candidates, target, 0, current1, 0, result1);

    sort(candidates.begin(), candidates.end());
    vector<int> current2;
    vector<vector<int>> result2;
    combinationSumPruned(candidates, target, 0, current2, 0, result2);

    cout << "unpruned: " << result1.size() << " results, " << callsUnpruned << " calls\n";
    cout << "pruned:   " << result2.size() << " results, " << callsPruned << " calls\n";
    cout << "solutions: ";
    for (auto &c : result2) {
        cout << "{ ";
        for (int x : c) cout << x << " ";
        cout << "} ";
    }
    cout << "\n";
    // both find the same solutions -- pruning changes HOW MUCH WORK it takes, never WHAT
    // the answer is. This is the core value proposition of pruning: same correctness,
    // fewer wasted branches explored

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. N-Queens — Constraint Satisfaction

### The Why
N-Queens is the classic example of backtracking with real constraint checking, not just "has this value been used" bookkeeping. It's also a clean demonstration of how a smart problem SETUP (placing one queen per row, by construction) can eliminate an entire category of conflicts before any checking code even runs.

### Core Mechanism
- Placing exactly one queen per row (by looping `row` from 0 to n-1 and choosing a column for each) automatically guarantees no two queens ever share a row — that conflict is eliminated by the structure of the recursion itself, not by a runtime check.
- What's left to check: **column conflicts** (two queens with the same column value) and **diagonal conflicts**. The diagonal check has an elegant O(1) form: two queens are on the same diagonal exactly when the absolute difference in their rows equals the absolute difference in their columns — this is because moving diagonally always changes row and column by the same amount.
- Choose a column for the current row only if `isSafe` confirms no conflict with any queen placed in an EARLIER row (later rows haven't been decided yet, so there's nothing to check against them).
- Notice there's no explicit "unchoose" line in the loop — `queenCol[row]` simply gets overwritten by the next iteration's choice, which is an equally valid (and slightly more efficient) form of undoing than an explicit reset.

### Common Pitfalls
- Checking for conflicts against ALL other queens including ones not yet placed — wasteful and, since unplaced rows haven't been decided, meaningless. Only check against rows strictly earlier than the current one.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// place queens one row at a time -- this alone eliminates the "same row" conflict entirely,
// since each row gets exactly one queen by construction. We still have to check columns
// and diagonals against every queen placed in EARLIER rows.
long long solutionCount = 0;

bool isSafe(vector<int>& queenCol, int row, int col) {
    for (int prevRow = 0; prevRow < row; prevRow++) {
        int prevCol = queenCol[prevRow];
        if (prevCol == col) return false;                              // same column
        if (abs(prevCol - col) == abs(prevRow - row)) return false;     // same diagonal
        // diagonal check works because on any diagonal, the row difference always equals
        // the column difference (in absolute value) -- a clean O(1) check per prior queen
    }
    return true;
}

void solveNQueens(int n, int row, vector<int>& queenCol) {
    if (row == n) {
        solutionCount++;
        return;
    }

    for (int col = 0; col < n; col++) {
        if (!isSafe(queenCol, row, col)) continue;   // PRUNE: skip unsafe placements entirely,
                                                        // never even recurse into them

        queenCol[row] = col;                          // CHOOSE
        solveNQueens(n, row + 1, queenCol);            // EXPLORE
        // no explicit "unchoose" needed here -- queenCol[row] just gets OVERWRITTEN by the
        // next iteration of this same loop, which is an equally valid form of undoing
    }
}

int main() {
    for (int n : {4, 6, 8}) {
        solutionCount = 0;
        vector<int> queenCol(n, -1);   // queenCol[row] = which column that row's queen sits in
        solveNQueens(n, 0, queenCol);
        cout << "n=" << n << " queens: " << solutionCount << " solutions\n";
    }
    // known values: n=4 -> 2, n=6 -> 4, n=8 -> 92 -- these confirm correctness

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. Word Search — Backtracking on a Grid

### The Why
This is where backtracking meets grid/graph traversal — a preview of the DFS you'll see formalized in `06_Graphs`. The bookkeeping is conceptually identical to `used[]` in permutations, just reshaped into a 2D `visited[][]` grid, because a path through a grid can't reuse the same cell twice.

### Core Mechanism
- At each grid cell, check three failure conditions before doing anything else: off the grid, already visited in this path, or the character doesn't match what the word needs next. Any one of these means this branch fails immediately.
- Choose: mark the current cell visited. Explore: recursively try all 4 directions (up, down, left, right) looking for the NEXT character in the word. Unchoose: mark the cell unvisited again — **regardless of whether this branch found the word or not** — because a cell used in one attempted path might be perfectly valid to use again starting from a different cell or a different path.
- The `||` short-circuit across the four directional recursive calls means the moment ANY direction succeeds, the remaining directions are never even attempted — a form of pruning that comes for free from how `||` evaluates in C++.

### Common Pitfalls
- Unmarking `visited[row][col]` only inside an `if (found)` branch instead of unconditionally — a cell that failed to lead anywhere from THIS starting point must still be released so a different starting point (or a different direction from the same neighbor) can try using it.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <string>
using namespace std;

// search a 2D grid for a word, moving to adjacent cells (up/down/left/right), never
// reusing the same cell twice within one attempt -- a grid is just a graph where backtracking
// needs to mark cells as "in use" and unmark them, exactly like used[] in permutations
bool search(vector<vector<char>>& grid, string& word, int row, int col, int idx,
            vector<vector<bool>>& visited) {
    if (idx == (int)word.size()) return true;   // matched every character -- found it

    int rows = grid.size(), cols = grid[0].size();
    if (row < 0 || row >= rows || col < 0 || col >= cols) return false;   // off the grid
    if (visited[row][col]) return false;                                  // already used in this path
    if (grid[row][col] != word[idx]) return false;                        // wrong character

    visited[row][col] = true;                                             // CHOOSE

    // EXPLORE all 4 directions
    bool found = search(grid, word, row + 1, col, idx + 1, visited) ||
                 search(grid, word, row - 1, col, idx + 1, visited) ||
                 search(grid, word, row, col + 1, idx + 1, visited) ||
                 search(grid, word, row, col - 1, idx + 1, visited);

    visited[row][col] = false;                                            // UNCHOOSE, whether
                                                                            // we found it via this
                                                                            // cell or not
    return found;
}

bool exists(vector<vector<char>>& grid, string word) {
    int rows = grid.size(), cols = grid[0].size();
    vector<vector<bool>> visited(rows, vector<bool>(cols, false));

    for (int r = 0; r < rows; r++) {
        for (int c = 0; c < cols; c++) {
            if (search(grid, word, r, c, 0, visited)) return true;   // try starting from every cell
        }
    }
    return false;
}

int main() {
    vector<vector<char>> grid{
        {'A','B','C','E'},
        {'S','F','C','S'},
        {'A','D','E','E'}
    };

    cout << "\"ABCCED\" found: " << exists(grid, "ABCCED") << "\n";   // true
    cout << "\"SEE\" found: " << exists(grid, "SEE") << "\n";         // true
    cout << "\"ABCB\" found: " << exists(grid, "ABCB") << "\n";       // false -- would need to reuse 'B'

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 7. When Backtracking Is (and Isn't) the Right Tool

### The Why
Backtracking with good pruning can turn a mathematically exponential problem into something that runs comfortably for realistic input sizes — but "comfortably" has a limit, and knowing roughly where that limit sits is what separates a solution you should actually submit from one that will time out no matter how clever the pruning is.

### Core Mechanism
- **Backtracking is the right tool when:** the problem explicitly asks for "all" solutions/arrangements/paths, OR when the search space — even though exponential in the worst case — is small enough in practice (small `n`, or heavily prunable) to finish in time. N-Queens for `n` up to the low 20s, word search on typical grid sizes, and subset/permutation problems on typically-sized inputs (n ≤ ~20 for subsets, since 2²⁰ ≈ 1 million) are all comfortably in this zone.
- **Backtracking is the WRONG tool when:** the problem only asks for a count, an optimum, or a yes/no answer, and there's overlapping substructure to exploit — that's a signal to look at dynamic programming instead (a preview of a later phase), which can turn some exponential backtracking problems into polynomial ones, the same way memoization turned exponential Fibonacci into linear Fibonacci in `02_Recursion`.
- **A rough sanity check before committing to backtracking:** estimate the size of the search tree (branching factor raised to the depth, adjusted for expected pruning) and compare it to what a typical time limit allows — as a rough rule of thumb, a modern CPU handles on the order of 10⁸–10⁹ simple operations per second, so a search tree with 10¹² unpruned leaves is not going to finish in a few seconds no matter how the code is written.

### Common Pitfalls
- Reaching for backtracking on a problem that's asking for a single optimal answer (not "all" answers) without first checking whether the sub-problems overlap — if they do, a DP approach will likely be dramatically faster, and backtracking (even well-pruned) risks being the wrong tool entirely, not just a slower version of the right one.


## 8. Phase Checkpoint, Cheat Sheet, and Answer Key

### Backtracking Pattern Cheat Sheet

| Problem type | Duplicate-avoidance mechanism | Typical complexity | Example |
|---|---|---|---|
| Permutations | `used[]` array (any order, no reuse) | O(n!) | All orderings of n items |
| Combinations | `start` index (only move forward) | O(2ⁿ) or O(C(n,k)) | Choose k of n |
| Subsets | include/exclude per element | O(2ⁿ) | All subsets |
| Constraint satisfaction | Problem-specific validity check (e.g. `isSafe`) | Varies, often pruned well below worst case | N-Queens, Sudoku |
| Grid/graph search | `visited[][]` grid | O(4^(path length)) unpruned, much less with pruning | Word search |

### Checkpoint Questions
1. What are the three steps of the backtracking template, and what specifically does "unchoose" undo?
2. Why does generating permutations need a `used[]` array, while generating combinations needs a `start` index instead?
3. In the pruned combination-sum example, why does the code use `break` instead of `continue` when a candidate would overshoot the target?
4. Explain why N-Queens' diagonal conflict check (`abs(prevCol - col) == abs(prevRow - row)`) works, in your own words.
5. In the word search grid backtracking, why must `visited[row][col]` be reset to `false` even on a branch that failed to find the word?
6. A problem asks for the MINIMUM number of coins to make change for a target amount (not all the ways — just the minimum count). Would you reach for backtracking here, or something else? Why?

### Answer Key
1. Choose (make a decision), explore (recurse into everything that follows), unchoose (undo the decision). "Unchoose" undoes whatever state change "choose" made — removing an element from a path, unmarking a cell as visited, freeing up a used value — so the next sibling branch starts from a clean, correct state.
2. Permutations care about ORDER — the same set of values in a different order is a different valid result — so you need to track which specific values have been used regardless of position, which is what `used[]` does. Combinations don't care about order, so a `start` index (only ever moving forward through the candidates) is what prevents generating the same set of values in multiple different orders as if they were distinct results.
3. Because the candidates are sorted first. If candidate `i` already overshoots the target when added to the current sum, every candidate AFTER `i` is guaranteed to be equal or larger (since the array is sorted), so they would overshoot too — `break` skips checking all of them, while `continue` would still (uselessly) check each one individually.
4. On any diagonal line, moving one step always changes both the row and the column by the same amount (in absolute value) — so two positions lie on the same diagonal exactly when the absolute difference between their rows equals the absolute difference between their columns.
5. Because a cell that didn't lead to a valid word from THIS particular attempted path might still be perfectly usable as part of a different path — either starting from a different cell entirely, or reached via a different direction from a different neighboring cell. Leaving it marked visited would incorrectly block those other valid attempts.
6. Something else — likely dynamic programming. "Minimum count" (an optimum, not "all the ways") strongly suggests overlapping sub-problems (the minimum coins for amount X is reused when computing the minimum for many larger amounts), which is exactly the situation where memoization/DP turns exponential backtracking into polynomial time, the same way it turned exponential Fibonacci into linear Fibonacci in `02_Recursion`.

### Mini Project
Implement a Sudoku validity-based backtracking solver for a SMALL constraint-satisfaction puzzle of your choosing (a simplified 4×4 Sudoku works well and keeps runtime instant):
1. Represent the board as a 2D grid, with `0` marking empty cells.
2. Write an `isValid(board, row, col, value)` check analogous to N-Queens' `isSafe` — checking the row, column, and the small sub-box for conflicts.
3. Backtrack: find the next empty cell, try each value 1 through 4, recurse if valid, undo if the recursive call fails to find a full solution.
4. Confirm your solver finds a correct solution on a puzzle with a few cells pre-filled.

This exercises: the full choose/explore/unchoose template, a genuine multi-part validity check (not just single-condition pruning), and connects directly back to N-Queens' pattern of "check constraints against everything decided so far."
